# Inference — X-Health Default Prediction

This notebook demonstrates how to use the trained model to predict default for new orders.

> **Prerequisite:** Run `02_modeling.ipynb` first to generate `models/pipeline_xgb.joblib`.

## 1. Setup

In [7]:
import sys
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

from src.predict import predict, predict_proba

## 2. The Prediction Function

```python
def predict(input_dict: dict) -> dict:
    """
    Receives input data and returns default prediction.
    """
```

- **Input:** dictionary with any subset of the model features
- **Output:** `{"default": 0}` or `{"default": 1}`
- Features not provided are **automatically imputed** using training statistics
- The pipeline handles all preprocessing internally — no manual transformation needed

## 3. Example A — Low-risk client

In [9]:
input_a = {
    "default_3months": 0,
    "ioi_36months": 30.0,
    "ioi_3months": 28.0,
    "valor_por_vencer": 5000.0,
    "valor_vencido": 0.0,
    "valor_quitado": 150000.0,
    "quant_protestos": 0,
    "valor_protestos": 0.0,
    "quant_acao_judicial": 0,
    "acao_judicial_valor": 0.0,
    "participacao_falencia_valor": 0.0,
    "dividas_vencidas_valor": 0.0,
    "dividas_vencidas_qtd": 0,
    "falencia_concordata_qtd": 0,
    "tipo_sociedade": "sociedade empresaria limitada",
    "opcao_tributaria": "simples nacional",
    "atividade_principal": "com de equipamentos de informatica",
    "forma_pagamento": "30/60/90",
    "valor_total_pedido": 25000.0,
    "month": 5,
    "year": 2021
}

print(predict(input_a, 0.64))
print(predict_proba(input_a, 0.64))

{'default': 0}
{'default': 0, 'probability': 0.4957}


## 4. Example B — High-risk client (realistic values based on dataset distribution)

> **Note:** Variables like `quant_protestos`, `dividas_vencidas_qtd` and `valor_vencido` are zero for over 85% of clients in the dataset.
> A realistic high-risk profile uses values in the **75th–95th percentile among non-zero cases** — not extreme outliers.
> Using values above the 99th percentile would push the model into a region with very few training examples, reducing its confidence.

In [12]:
# Real values from a client in the test set that actually defaulted (y_proba=0.998)
# Key insight: the strongest signal is NOT Serasa data, but internal X-Health history:
#   - default_3months=4: already defaulted 4 times in the last 3 months
#   - ioi_3months=6.2: placing orders every 6 days (very high frequency)
#   - valor_por_vencer=77k: large amount still due
input_b = {
    "default_3months": 4,
    "ioi_36months": 10.93,
    "ioi_3months": 6.21,
    "valor_por_vencer": 77610.74,
    "valor_vencido": 75.20,
    "valor_quitado": 1450553.24,
    "quant_protestos": 0,
    "valor_protestos": 40.48,
    "quant_acao_judicial": 0,
    "acao_judicial_valor": 0.0,
    "participacao_falencia_valor": 0.0,
    "dividas_vencidas_valor": 0.0,
    "dividas_vencidas_qtd": 0,
    "falencia_concordata_qtd": 0,
    "tipo_sociedade": "sociedade empresaria limitada",
    "opcao_tributaria": "simples nacional",
    "atividade_principal": "com de telefones e equip p/ comunicacoes",
    "forma_pagamento": "30/60/90/120",
    "valor_total_pedido": 16194.23,
    "month": 12,
    "year": 2019
}


print(predict(input_b, 0.64))
print(predict_proba(input_b, 0.64))

{'default': 1}
{'default': 1, 'probability': 0.9981}


## 5. Example C — Partial data (only a few features known)

In [5]:
# Missing features are imputed automatically using training medians/modes
input_c = {
    "ioi_3months": 3.0,
    "valor_vencido": 125000.0,
    "valor_total_pedido": 35000.0
}

print(predict(input_c, 0.64))
print(predict_proba(input_c, 0.64))

{'default': 0}
{'default': 0, 'probability': 0.5209}


## 6. Batch Prediction

In [11]:
batch = [input_a, input_b, input_c]
results = [predict_proba(inp) for inp in batch]

pd.DataFrame(results)

,default,probability
0,0,0.4957
1,0,0.1547
2,1,0.5209


## 7. How to Integrate Into a Real System

```python
from src.predict import predict, predict_proba

# Receive order data from any source (API, database, form)
order = fetch_order_data(order_id=12345)

# Run prediction
result = predict_proba(order)

# Route decision based on probability
if result['probability'] > 0.4:   # custom threshold defined by finance team
    flag_for_credit_review(order_id)
else:
    approve_order(order_id)
```

**Note on threshold:** The default threshold is 0.5 (`predict()`), but the finance team can use `predict_proba()` and define their own cutoff based on acceptable risk — lower threshold = catches more defaults, but blocks more good clients.